# 1 · Processamento de Dados — dataset unificado (PySpark)

**Objetivo:** transformar os 3 arquivos brutos em uma **tabela de modelagem (cliente × onda)**
que sustenta tanto o modelo de resposta quanto a leitura causal do envio.

## Desenho (decisões e alternativas descartadas)

| Decisão | Justificativa | Alternativa descartada |
|---|---|---|
| **Unidade = (cliente × onda)** | Os envios ocorrem em 6 ondas (dias 0,7,14,17,21,24) e cada cliente recebe **no máx. 1 oferta por onda** (verificado adiante). A linha fica não-ambígua: tratado (com atributos da oferta) ou controle. | (cliente × oferta recebida): sem linhas de controle não se estima efeito do envio; 56% das janelas se sobrepõem, atribuição ambígua. |
| **Tratamento W = envio na onda** | É a alavanca que o negócio controla. ~25% da base não recebe nada em cada onda e o *balance check* (adiante) sugere aleatorização → controle utilizável. | W = visualizou: é **pós-tratamento** (mediador); condicionar nele embute viés de seleção. |
| **Controle limpo** = não recebeu na onda **e** sem janela anterior ativa | Cliente sem oferta vigente é contrafactual do "não enviar". | Usar todos os não-recebedores: contaminados por ofertas anteriores ainda ativas. |
| **Target resposta `y_response` = viu E usou** (`t_view ≤ t_comp`, na validade), **só para bogo/discount** | "Usar" = `offer completed` (resgate). Sem exigir visualização **anterior**, 16,3% das instâncias seriam falsos sucessos (auto-resgate: 9,4% sem ver + 6,9% viu depois). **Informational fica nulo**: não tem evento de resgate, e "comprou após ver" ocorre 47,6% das vezes sem oferta alguma — mede atividade basal, não resposta. Seu efeito é medido pelo gasto vs controle. | `completed` puro: contamina o target com compras que ocorreriam de qualquer forma. Forçar um alvo para informational: mistura evento impossível-sem-oferta (resgate) com evento que acontece sozinho (compra). |
| **Outcome contínuo = gasto em horizontes fixos** (3/4/5/7/10d) | Permite contraste tratado × controle com **janelas do mesmo tamanho** (controle não tem `duration`). | Gasto na janela da oferta apenas: sem equivalente no controle. |
| **Features estritamente pré-onda** (`t < dia da onda`) | Fecha vazamento temporal. Onda 0 fica sem histórico → flag `has_history`. | Agregados do período todo: vazam o próprio desfecho. |
| **Split temporal** (treino ondas 0–14, teste 17–24; aplicado no NB2) | Simula decisão real: treinar no passado, decidir no futuro. | Split aleatório: vaza o futuro e superestima performance. |

## Premissas
1. `age == 118` é sentinela de perfil incompleto (coincide 100% com nulos de `gender`/`credit_card_limit`).
2. Janela de validade = `[t_recv, t_recv + duration]` dias; `time_since_test_start` em dias (0–29.75).
3. `offer completed` = cupom usado (resgate ao atingir `min_value` na validade).
4. Envio nas ondas ≈ aleatorizado (suportado pelo balance check; validação definitiva = A/B).


## Setup (dual-mode: local ou Databricks)
O notebook roda **sem alterações** em ambiente local ou no Databricks (Free Edition/
Community): detecta o ambiente, obtém a sessão Spark via `getOrCreate` e, se os
arquivos brutos não existirem no disco, **baixa o tar.gz da URL pública do S3**.

In [1]:
import os, sys, glob, tarfile, tempfile, urllib.request
from pathlib import Path
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import types as T

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()   # sessão provida pela plataforma
else:
    # Windows/local: aponta os workers Python do Spark para o interpretador atual
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    # heap do driver precisa ser definido ANTES da JVM subir (config no builder é ignorada)
    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 4g pyspark-shell")
    spark = (SparkSession.builder.master("local[*]").appName("ifood-nb1")
             .config("spark.sql.shuffle.partitions", "8")
             .config("spark.ui.enabled", "false").getOrCreate())
    # sparkContext não existe no serverless (Spark Connect) — só em modo local
    spark.sparkContext.setLogLevel("ERROR")

CWD = Path(os.getcwd())
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_URL = "https://data-architect-test-source.s3.sa-east-1.amazonaws.com/ds-technical-evaluation-data.tar.gz"
print("Ambiente:", "Databricks" if IS_DATABRICKS else "local", "| Spark", spark.version)

C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Ambiente: local | Spark 4.2.0


In [2]:
def ensure_raw_data():
    """Retorna dir com offers/profile/transactions.json; baixa do S3 se preciso."""
    raw = ROOT / "data" / "raw"
    if (raw / "transactions.json").exists():
        return raw
    tmp = Path(tempfile.gettempdir()) / "ifood_raw"
    if not (tmp / "transactions.json").exists():
        tmp.mkdir(parents=True, exist_ok=True)
        tgz = tmp / "data.tar.gz"
        print("Baixando dados de", DATA_URL)
        urllib.request.urlretrieve(DATA_URL, tgz)
        with tarfile.open(tgz) as t:
            t.extractall(tmp)
        for f in glob.glob(str(tmp / "**" / "*.json"), recursive=True):
            Path(f).replace(tmp / Path(f).name)
    return tmp

RAW = ensure_raw_data()
print("Dados em:", RAW)

Dados em: C:\Users\Maxel\OneDrive\Documentos\github\ifood-case\data\raw


## Ingestão
Parse do campo aninhado `value` (`offer id` com espaço em received/viewed,
`offer_id` em completed, `amount` em transações) feito em pandas — caminho de
ingestão único que funciona igual local e no serverless do Databricks — e conversão
para Spark **com schema explícito**. Todo o processamento pesado é PySpark.

In [3]:
offers_pd = pd.read_json(RAW / "offers.json")
profile_pd = pd.read_json(RAW / "profile.json")
tx_pd = pd.read_json(RAW / "transactions.json")

tx_pd["offer_id"] = tx_pd["value"].apply(lambda d: d.get("offer id") or d.get("offer_id"))
tx_pd["amount"] = pd.to_numeric(tx_pd["value"].apply(lambda d: d.get("amount")))
tx_pd["reward"] = pd.to_numeric(tx_pd["value"].apply(lambda d: d.get("reward")))
tx_flat = tx_pd[["account_id", "event", "offer_id", "amount", "reward",
                 "time_since_test_start"]].rename(columns={"time_since_test_start": "t"})
offers_pd["channels_str"] = offers_pd["channels"].apply(",".join)
profile_pd["registered_on"] = profile_pd["registered_on"].astype(str)

def none_ify(df):
    return df.astype(object).where(df.notna(), None)

tx_s = spark.createDataFrame(none_ify(tx_flat), schema=T.StructType([
    T.StructField("account_id", T.StringType()), T.StructField("event", T.StringType()),
    T.StructField("offer_id", T.StringType()), T.StructField("amount", T.DoubleType()),
    T.StructField("reward", T.DoubleType()), T.StructField("t", T.DoubleType())]))
offers_s = spark.createDataFrame(
    none_ify(offers_pd[["id", "offer_type", "min_value", "discount_value", "duration", "channels_str"]]),
    schema=T.StructType([
        T.StructField("offer_id", T.StringType()), T.StructField("offer_type", T.StringType()),
        T.StructField("min_value", T.DoubleType()), T.StructField("discount_value", T.DoubleType()),
        T.StructField("duration", T.DoubleType()), T.StructField("channels_str", T.StringType())]))
profile_s = spark.createDataFrame(
    none_ify(profile_pd[["id", "age", "gender", "credit_card_limit", "registered_on"]]),
    schema=T.StructType([
        T.StructField("account_id", T.StringType()), T.StructField("age", T.DoubleType()),
        T.StructField("gender", T.StringType()), T.StructField("credit_card_limit", T.DoubleType()),
        T.StructField("registered_on", T.StringType())]))
# cache: os DFs vêm de coleções locais (LocalRelation); sem cache, cada join
# re-embute e recomputa os dados no plano. Só em modo local — persist/cache
# não é suportado no compute serverless do Databricks.
if not IS_DATABRICKS:
    tx_s = tx_s.repartition(8).cache()
    offers_s = offers_s.cache()
    profile_s = profile_s.cache()
print("tx:", tx_s.count(), "| offers:", offers_s.count(), "| profile:", profile_s.count())

C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. 

tx: 306534 | offers: 10 | profile: 17000


## Limpeza — perfil e ofertas

In [4]:
CHANNELS = ["web", "email", "mobile", "social"]
offers_c = offers_s.withColumn("channels", F.split("channels_str", ","))
for ch in CHANNELS:
    offers_c = offers_c.withColumn(f"ch_{ch}", F.array_contains("channels", ch).cast("int"))
offers_c = offers_c.withColumn("n_channels", sum(F.col(f"ch_{c}") for c in CHANNELS)) \
                   .drop("channels", "channels_str")

ref_date = profile_s.select(F.max(F.to_date("registered_on", "yyyyMMdd"))).first()[0]
profile_c = (profile_s
    .withColumn("incomplete_profile", (F.col("age") == 118).cast("int"))
    .withColumn("age", F.when(F.col("age") == 118, None).otherwise(F.col("age")))
    .withColumn("gender", F.coalesce("gender", F.lit("unknown")))
    .withColumn("account_age_days",
                F.datediff(F.lit(ref_date), F.to_date("registered_on", "yyyyMMdd")))
    .drop("registered_on"))
profile_c.show(3)

+--------------------+----+-------+-----------------+------------------+----------------+
|          account_id| age| gender|credit_card_limit|incomplete_profile|account_age_days|
+--------------------+----+-------+-----------------+------------------+----------------+
|68be06ca386d4c319...|NULL|unknown|             NULL|                 1|             529|
|0610b486422d4921a...|55.0|      F|         112000.0|                 0|             376|
|38fe809add3b4fcf9...|NULL|unknown|             NULL|                 1|              14|
+--------------------+----+-------+-----------------+------------------+----------------+
only showing top 3 rows


## Ondas de envio e verificação da unidade
Confirmamos empiricamente: 6 ondas e **no máximo 1 oferta por cliente por onda** —
o que valida a unidade (cliente × onda).

In [5]:
events = {e: tx_s.filter(F.col("event") == e).drop("event")
          for e in ["offer received", "offer viewed", "offer completed", "transaction"]}
rec = events["offer received"].select("account_id", "offer_id", F.col("t").alias("t_recv"))
vie = events["offer viewed"].select("account_id", "offer_id", F.col("t").alias("t_view"))
com = events["offer completed"].select("account_id", "offer_id", F.col("t").alias("t_comp"))
trans = events["transaction"].select("account_id", F.col("t").alias("t_tx"), "amount")

waves = [r[0] for r in rec.select("t_recv").distinct().orderBy("t_recv").collect()]
print("ondas:", waves)
dup = (rec.groupBy("account_id", "t_recv").count().filter("count > 1").count())
assert len(waves) == 6 and dup == 0, "premissa violada"
print("máx. 1 oferta por (cliente, onda): OK")

ondas: [0.0, 7.0, 14.0, 17.0, 21.0, 24.0]


máx. 1 oferta por (cliente, onda): OK


## Grade (cliente × onda) + tratamento
17.000 clientes × 6 ondas = 102.000 linhas. `W=1` se recebeu oferta na onda.
`n_active_offers` = janelas de ondas anteriores ainda vigentes no dia do envio —
define o **controle limpo** (`W=0` e nenhuma oferta ativa).

In [6]:
waves_s = spark.createDataFrame([(float(w),) for w in waves], ["wave"])
grid = profile_c.select("account_id").crossJoin(waves_s)

inst = (rec.join(offers_c, "offer_id")
           .withColumn("t_end", F.col("t_recv") + F.col("duration")))

active = (grid.join(inst.select("account_id", F.col("t_recv").alias("t0"), "t_end"), "account_id")
              .filter((F.col("t0") < F.col("wave")) & (F.col("t_end") > F.col("wave")))
              .groupBy("account_id", "wave").agg(F.count("*").alias("n_active_offers")))

grid = (grid.join(inst.withColumnRenamed("t_recv", "wave"), ["account_id", "wave"], "left")
            .join(active, ["account_id", "wave"], "left")
            .fillna({"n_active_offers": 0})
            .withColumn("W", F.col("offer_id").isNotNull().cast("int"))
            .withColumn("control_clean",
                        ((F.col("W") == 0) & (F.col("n_active_offers") == 0)).cast("int")))
if not IS_DATABRICKS:
    grid = grid.cache()   # persist não é suportado no serverless
print("grade:", grid.count())
grid.groupBy("wave").agg(F.sum("W").alias("tratados"),
                         F.sum(F.when(F.col("W") == 0, 1).otherwise(0)).alias("nao_recebeu"),
                         F.sum("control_clean").alias("controle_limpo")).orderBy("wave").show()

grade: 102000


+----+--------+-----------+--------------+
|wave|tratados|nao_recebeu|controle_limpo|
+----+--------+-----------+--------------+
| 0.0|   12650|       4350|          4350|
| 7.0|   12669|       4331|          3678|
|14.0|   12711|       4289|          3622|
|17.0|   12778|       4222|          1362|
|21.0|   12704|       4296|          1436|
|24.0|   12765|       4235|          1182|
+----+--------+-----------+--------------+



## Atribuição na janela da oferta (tratados)
`first_view`/`first_comp` do **mesmo offer_id** dentro de `[t_recv, t_end]`.
O flag `view_before_comp` implementa a ordem exigida pelo target.

In [7]:
base_t = grid.filter("W = 1").select("account_id", "wave", "offer_id", "t_end", "offer_type",
                                     "discount_value")

fv = (base_t.join(vie, ["account_id", "offer_id"])
            .filter((F.col("t_view") >= F.col("wave")) & (F.col("t_view") <= F.col("t_end")))
            .groupBy("account_id", "wave").agg(F.min("t_view").alias("first_view")))
fc = (base_t.join(com, ["account_id", "offer_id"])
            .filter((F.col("t_comp") >= F.col("wave")) & (F.col("t_comp") <= F.col("t_end")))
            .groupBy("account_id", "wave").agg(F.min("t_comp").alias("first_comp")))

attrib = (base_t.join(fv, ["account_id", "wave"], "left")
                .join(fc, ["account_id", "wave"], "left")
                .withColumn("viewed", F.col("first_view").isNotNull().cast("int"))
                .withColumn("completed", F.col("first_comp").isNotNull().cast("int"))
                .withColumn("view_before_comp",
                            ((F.col("viewed") == 1) & (F.col("completed") == 1) &
                             (F.col("first_view") <= F.col("first_comp"))).cast("int"))
                .withColumn("reward_paid",
                            F.when(F.col("completed") == 1, F.col("discount_value")).otherwise(0.0)))

# informational: transação APÓS a visualização, dentro da janela
tx_after_view = (attrib.filter("offer_type = 'informational' and viewed = 1")
    .select("account_id", "wave", "first_view", "t_end")
    .join(trans, "account_id")
    .filter((F.col("t_tx") >= F.col("first_view")) & (F.col("t_tx") <= F.col("t_end")))
    .groupBy("account_id", "wave").agg(F.count("*").alias("n_tx_after_view")))

attrib = attrib.join(tx_after_view, ["account_id", "wave"], "left") \
               .fillna({"n_tx_after_view": 0})

# y_response (viu antes de usar) só existe para bogo/discount; informational não tem
# evento de resgate. Para referência, mostramos também quantos informational
# transacionaram após ver — mas isso NÃO vira alvo (ver justificativa acima).
attrib.groupBy("offer_type").agg(
    F.mean("viewed").alias("view_rate"),
    F.mean("completed").alias("comp_rate"),
    # cada métrica só é exibida onde é definida — evita 0.0 que parece medição
    F.mean(F.when(F.col("offer_type") != "informational",
                  F.col("view_before_comp"))).alias("y_response"),
    F.mean(F.when(F.col("offer_type") == "informational",
                  ((F.col("viewed") == 1) & (F.col("n_tx_after_view") > 0)).cast("int"))
     ).alias("viu_e_transacionou")).show()

+-------------+------------------+------------------+-------------------+------------------+
|   offer_type|         view_rate|         comp_rate|         y_response|viu_e_transacionou|
+-------------+------------------+------------------+-------------------+------------------+
|     discount| 0.705857315915267|0.5880889238123301|0.41243492780669877|              NULL|
|informational|0.6540203478831638|               0.0|               NULL|0.3910731867410568|
|         bogo|0.8318961277418931|0.5137545493294862|0.36329059969179317|              NULL|
+-------------+------------------+------------------+-------------------+------------------+



## Outcomes em horizontes fixos (todas as linhas, tratado e controle)
Gasto e nº de transações em `[onda, onda+h]` para h ∈ {3,4,5,7,10} — janelas
idênticas para tratados e controles, viabilizando o contraste causal. Para tratados,
também o gasto na **janela da própria oferta** `[onda, t_end]`.

In [8]:
HORIZONS = [3, 4, 5, 7, 10]
tx_out = (grid.select("account_id", "wave", "t_end")
              .join(trans, "account_id")
              .filter((F.col("t_tx") >= F.col("wave")) & (F.col("t_tx") <= F.col("wave") + 10)))
aggs = []
for h in HORIZONS:
    cond = F.col("t_tx") <= F.col("wave") + h
    aggs += [F.sum(F.when(cond, F.col("amount")).otherwise(0.0)).alias(f"spend_h{h}"),
             F.sum(F.when(cond, 1).otherwise(0)).alias(f"n_tx_h{h}")]
aggs += [F.sum(F.when(F.col("t_tx") <= F.col("t_end"), F.col("amount")).otherwise(0.0)).alias("spend_window"),
         F.sum(F.when(F.col("t_tx") <= F.col("t_end"), 1).otherwise(0)).alias("n_tx_window")]
outcomes = tx_out.groupBy("account_id", "wave").agg(*aggs)
print("linhas com alguma transação no horizonte:", outcomes.count())

linhas com alguma transação no horizonte: 88203


## Features estritamente pré-onda (`t < onda`)
RFM transacional + histórico de ofertas do cliente **antes** do envio. Onda 0 não
tem histórico → `has_history = 0` (mantida com flag; descartá-la custaria 1/6 dos
tratados e o controle mais limpo).

In [9]:
tx_pre = (grid.select("account_id", "wave").join(trans, "account_id")
              .filter(F.col("t_tx") < F.col("wave")))
rfm = tx_pre.groupBy("account_id", "wave").agg(
    F.count("*").alias("n_tx_pre"),
    F.round(F.sum("amount"), 2).alias("spend_pre"),
    F.round(F.avg("amount"), 2).alias("avg_ticket_pre"),
    F.countDistinct(F.floor("t_tx")).alias("active_days_pre"),
    (F.first("wave") - F.max("t_tx")).alias("recency_days"))

hist_r = (grid.select("account_id", "wave").join(rec, "account_id")
              .filter(F.col("t_recv") < F.col("wave"))
              .groupBy("account_id", "wave").agg(F.count("*").alias("n_received_pre")))
hist_v = (grid.select("account_id", "wave").join(vie, "account_id")
              .filter(F.col("t_view") < F.col("wave"))
              .groupBy("account_id", "wave").agg(F.count("*").alias("n_viewed_pre")))
hist_c = (grid.select("account_id", "wave").join(com, "account_id")
              .filter(F.col("t_comp") < F.col("wave"))
              .groupBy("account_id", "wave").agg(F.count("*").alias("n_completed_pre")))
print("features pré-onda prontas")

features pré-onda prontas


## Montagem da tabela de modelagem

In [10]:
modeling = (grid
    .join(attrib.select("account_id", "wave", "first_view", "first_comp", "viewed",
                        "completed", "view_before_comp", "reward_paid", "n_tx_after_view"),
          ["account_id", "wave"], "left")
    .join(outcomes, ["account_id", "wave"], "left")
    .join(rfm, ["account_id", "wave"], "left")
    .join(hist_r, ["account_id", "wave"], "left")
    .join(hist_v, ["account_id", "wave"], "left")
    .join(hist_c, ["account_id", "wave"], "left")
    .join(profile_c, "account_id", "left")
    .fillna({"viewed": 0, "completed": 0, "view_before_comp": 0, "reward_paid": 0.0,
             "n_tx_after_view": 0, "spend_window": 0.0, "n_tx_window": 0,
             "n_tx_pre": 0, "spend_pre": 0.0, "avg_ticket_pre": 0.0, "active_days_pre": 0,
             "n_received_pre": 0, "n_viewed_pre": 0, "n_completed_pre": 0,
             **{f"spend_h{h}": 0.0 for h in HORIZONS}, **{f"n_tx_h{h}": 0 for h in HORIZONS}})
    .withColumn("has_history", (F.col("wave") > 0).cast("int"))
    .withColumn("view_rate_pre",
                F.when(F.col("n_received_pre") > 0,
                       F.col("n_viewed_pre") / F.col("n_received_pre")).otherwise(None))
    .withColumn("comp_rate_pre",
                F.when(F.col("n_received_pre") > 0,
                       F.col("n_completed_pre") / F.col("n_received_pre")).otherwise(None))
    # Target resposta: viu E usou o cupom (com ordem). NULO para informational —
    # esse tipo não tem evento de resgate, e "comprou após ver" acontece 47,6% das
    # vezes sem oferta nenhuma (baseline do controle), então não é alvo comparável.
    # O efeito de informational é medido pelo gasto vs controle (NB2), não aqui.
    .withColumn("y_response",
                F.when(F.col("offer_type") == "informational", F.lit(None).cast("int"))
                 .when(F.col("W") == 0, F.lit(0))
                 .otherwise(F.col("view_before_comp")))
    # outcome líquido na janela da oferta (tratados)
    .withColumn("y_net_window",
                F.when(F.col("W") == 1, F.col("spend_window") - F.col("reward_paid"))))

n_rows = modeling.count()
print("tabela de modelagem:", n_rows, "linhas x", len(modeling.columns), "colunas")
modeling.groupBy("W").agg(
    F.count("*").alias("n"),
    F.count("y_response").alias("n_com_alvo"),   # exclui informational (nulo)
    F.mean("y_response").alias("y_resp"),
    F.mean("spend_h7").alias("spend_h7")).show()

tabela de modelagem: 102000 linhas x 53 colunas


+---+-----+----------+-------------------+------------------+
|  W|    n|n_com_alvo|             y_resp|          spend_h7|
+---+-----+----------+-------------------+------------------+
|  1|76277|     61042|0.38788047573801643|28.190644755299783|
|  0|25723|     25723|                0.0|18.907922481825604|
+---+-----+----------+-------------------+------------------+



## Sanity checks
1. Batem com a EDA estrutural (funil com ordem, controle limpo por onda)?
2. **Balance check** (onda 17): tratados vs controle limpo em features pré-onda —
   diferenças pequenas suportam a premissa de envio ~aleatório.

In [11]:
bd = modeling.filter("W = 1 and offer_type in ('bogo','discount')")
tot = bd.count()
print(f"instâncias bogo/discount: {tot:,}")
for lbl, cond in [("viu e usou (ordem ok)", "view_before_comp = 1"),
                  ("completou SEM ver", "completed = 1 and viewed = 0"),
                  ("completou antes de ver", "completed = 1 and viewed = 1 and view_before_comp = 0")]:
    v = bd.filter(cond).count()
    print(f"  {lbl:24s}: {v:6,} ({v/tot:6.1%})")

instâncias bogo/discount: 61,042


  viu e usou (ordem ok)   : 23,677 ( 38.8%)


  completou SEM ver       :  5,734 (  9.4%)


  completou antes de ver  :  4,220 (  6.9%)


In [12]:
bal = (modeling.filter("wave = 17 and (W = 1 or control_clean = 1)")
    .groupBy("W").agg(F.count("*").alias("n"),
                      F.round(F.mean("spend_pre"), 2).alias("spend_pre"),
                      F.round(F.mean("n_tx_pre"), 2).alias("n_tx_pre"),
                      F.round(F.mean("age"), 1).alias("age"),
                      F.round(F.mean("incomplete_profile"), 3).alias("perfil_incompleto")))
bal.orderBy("W").show()

+---+-----+---------+--------+----+-----------------+
|  W|    n|spend_pre|n_tx_pre| age|perfil_incompleto|
+---+-----+---------+--------+----+-----------------+
|  0| 1362|    47.57|    3.77|55.0|            0.138|
|  1|12778|    51.78|    4.11|54.4|            0.129|
+---+-----+---------+--------+----+-----------------+



## Persistência
Local: parquet em `data/processed/` (consumido pelo NB2). No Databricks:
tabelas gerenciadas no Unity Catalog (`saveAsTable` executa no cluster —
`toPandas()` de um plano grande esbarra em limitação de serialização do
Spark Connect no serverless).

In [13]:
if IS_DATABRICKS:
    modeling.write.mode("overwrite").saveAsTable("ifood_modeling_table")
    offers_c.write.mode("overwrite").saveAsTable("ifood_offers")
    print("salvo como tabelas: ifood_modeling_table / ifood_offers (catálogo padrão)")
    print("linhas:", spark.table("ifood_modeling_table").count())
else:
    out_dir = ROOT / "data" / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    modeling_pd = modeling.toPandas()
    modeling_pd.to_parquet(out_dir / "modeling_table.parquet", index=False)
    offers_c.toPandas().to_parquet(out_dir / "offers.parquet", index=False)
    print("salvo em", out_dir)
    print(modeling_pd.shape)

C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


salvo em C:\Users\Maxel\OneDrive\Documentos\github\ifood-case\data\processed
(102000, 53)


C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Resumo
- **Saída:** `modeling_table.parquet` — 102.000 linhas (cliente × onda): ~76,3k
  tratadas (`W=1`, com atributos da oferta e atribuição na janela) e ~25,7k de
  controle (~15,6k limpas), com outcomes em horizontes fixos e features pré-onda.
- **Targets:** `y_response` (viu E usou, com ordem) — definido apenas para
  bogo/discount, **nulo para informational**; e `y_net_window` / `spend_h*`
  (impacto contínuo), disponíveis para **todas** as linhas, inclusive controle.
- **Por que dois targets:** o de resposta serve para entender *quem usa cupom* e só
  faz sentido onde existe resgate; o contínuo é o que sustenta a decisão de envio,
  porque pode ser comparado contra o grupo de controle — inclusive para
  informational, cujo efeito só aparece como variação de gasto.
- **Próximo (NB2):** split temporal por onda, modelo de resposta (bogo/discount),
  contraste causal tratado × controle limpo e política de envio.

In [14]:
spark.stop() if not IS_DATABRICKS else None